# Airline Sentiment Analysis

Notebook عملي لتحليل مشاعر تغريدات شركات الطيران باستخدام `pandas`, `nltk` و`scikit-learn`.

**المسار:** تحميل الداتا → فحصها → تنظيف النص → حذف stopwords → TF-IDF → train/test split → Logistic Regression.

## 1) جهّز البيئة

شغّل الأمر التالي مرة واحدة في Terminal إذا كانت المكتبات غير مثبتة:

```bash
pip install pandas scikit-learn nltk matplotlib jupyter
jupyter notebook
```

In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

pd.set_option('display.max_colwidth', 120)

## 2) نزّل الداتاسة

نزّل `Tweets.csv` من [Kaggle](https://www.kaggle.com/datasets/crowdflower/twitter-airline-sentiment) وضعه في نفس فولدر هذا الـ notebook.

In [ ]:
df = pd.read_csv('Tweets.csv')
print(f'Rows: {len(df):,} | Columns: {len(df.columns)}')
df.head()

## 3) شوف الداتا بنفسك

نطبع أول صفوف ونشوف عدد التغريدات في كل فئة.

In [ ]:
display(df.head())
sentiment_counts = df['airline_sentiment'].value_counts()
display(sentiment_counts.to_frame('count'))

sentiment_counts.sort_values().plot(kind='barh', color=['#f19b91', '#f0d783', '#9ce4c5'])
plt.title('Airline sentiment distribution')
plt.xlabel('Number of tweets')
plt.ylabel('Sentiment')
plt.show()

## 4) نظّف النص بنفسك

نحوّل النص لحروف صغيرة ونزيل الروابط والـ mentions. نحتفظ بباقي علامات الترقيم في هذه المرحلة حتى نرى الفرق بوضوح.

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    return text.strip()

df['clean_text'] = df['text'].apply(clean_text)

for before, after in zip(df['text'].head(3), df['clean_text'].head(3)):
    print('BEFORE:', before)
    print('AFTER :', after)
    print('-' * 80)

## 5) احذف stopwords مع الحفاظ على `not` و`no`

In [ ]:
import nltk
nltk.download('stopwords')

stop_words = set(stopwords.words('english')) - {'not', 'no'}

def remove_stopwords(text):
    words = text.split()
    return ' '.join(word for word in words if word not in stop_words)

df['clean_text'] = df['clean_text'].apply(remove_stopwords)
print('قبل حذف stopwords:')
print('this is not a good flight')
print('بعد حذف stopwords مع الحفاظ على not/no:')
print(remove_stopwords('this is not a good flight'))

## 6) حوّل النص لأرقام باستخدام TF-IDF

نستخدم unigram وbigram ونحتفظ بأهم 5,000 ميزة.

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X = vectorizer.fit_transform(df['clean_text'])
y = df['airline_sentiment']

print('X shape:', X.shape)
print('First 10 feature names:', vectorizer.get_feature_names_out()[:10])
print('First row, first 10 TF-IDF values:', X[0, :10].toarray())

## 7) قسّم train / test

`stratify` يحافظ على نفس نسبة الفئات في مجموعتي التدريب والاختبار.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('X_train:', X_train.shape)
print('X_test :', X_test.shape)

## 8) درّب Logistic Regression

لو ظهر `ConvergenceWarning`، غيّر `max_iter` إلى `2000`.

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

predictions = model.predict(X_test)
print(f'Test accuracy: {accuracy_score(y_test, predictions):.3f}')
print(classification_report(y_test, predictions))

ConfusionMatrixDisplay.from_predictions(y_test, predictions, cmap='GnBu')
plt.title('Confusion matrix')
plt.show()

## جرّب تنبؤًا جديدًا

استخدم نفس `vectorizer` ثم مرّر النص للموديل.

In [ ]:
new_tweet = 'I had a great flight and the crew was amazing!'
new_tweet_clean = remove_stopwords(clean_text(new_tweet))
new_tweet_vector = vectorizer.transform([new_tweet_clean])
print('Prediction:', model.predict(new_tweet_vector)[0])